In [53]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
# =========================================================
# CREATE PROJECT FOLDER STRUCTURE
# =========================================================

import os

PROJECT_ROOT = "/content/drive/MyDrive/PersonalFashionStylistV2"

folders = [

    # Main folders
    "datasets",
    "models",
    "embeddings",
    "faiss",
    "notebooks",
    "outputs",
    "src",

    # Dataset folders
    "datasets/ecommerce",
    "datasets/deepfashion_subset",
    "datasets/fashion_products",
    "datasets/processed",
    "datasets/metadata",

    # Model folders
    "models/convnext_tiny",
    "models/checkpoints",
    "models/label_maps",
]

for folder in folders:
    path = os.path.join(PROJECT_ROOT, folder)
    os.makedirs(path, exist_ok=True)

print("✅ Project folder structure created successfully.")

✅ Project folder structure created successfully.


In [55]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

import os
import json
import random
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [56]:
# =========================================================
# GLOBAL CONFIG
# =========================================================

CONFIG = {

    # Paths
    "project_root": PROJECT_ROOT,

    # Random seed
    "seed": 42,

    # Image settings
    "img_size": 224,

    # Dataset split
    "train_ratio": 0.7,
    "val_ratio": 0.15,
    "test_ratio": 0.15,

    # Supported image formats
    "image_extensions": [".jpg", ".jpeg", ".png", ".webp"]
}

print("✅ Config initialized.")

✅ Config initialized.


In [57]:
# =========================================================
# SET RANDOM SEED
# =========================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

set_seed(CONFIG["seed"])

print("✅ Random seed fixed.")

✅ Random seed fixed.


In [58]:
# =========================================================
# DOWNLOAD ECOMMERCE MEN CLOTHING DATASET
# =========================================================

import kagglehub

# Download latest version
ecommerce_path = kagglehub.dataset_download("prashantsharma526/e-commerce-mens-clothing-dataset")


print("✅ Ecommerce dataset downloaded.")
print("Path:", ecommerce_path)

Using Colab cache for faster access to the 'e-commerce-mens-clothing-dataset' dataset.
✅ Ecommerce dataset downloaded.
Path: /kaggle/input/e-commerce-mens-clothing-dataset


In [59]:
# =========================================================
# INSPECT ECOMMERCE DATASET STRUCTURE
# =========================================================

for root, dirs, files in os.walk(ecommerce_path):

    print(f"\n📂 {root}")

    if len(files) > 0:
        print("Sample files:", files[:5])

    if len(dirs) > 0:
        print("Subfolders:", dirs[:5])

    # limit excessive printing
    if root.count(os.sep) > 3:
        break


📂 /kaggle/input/e-commerce-mens-clothing-dataset
Sample files: ['readme.md', 'Classes.txt']
Subfolders: ['sample_images', 'dataset_clean']

📂 /kaggle/input/e-commerce-mens-clothing-dataset/sample_images
Subfolders: ['cargos', 'formal_pants', 'casual_shirts', 'printed_hoodies', 'solid_tshirts']


In [60]:
# =========================================================
# DOWNLOAD FASHION PRODUCT DATASET
# =========================================================

fashion_products_path = kagglehub.dataset_download(
    "paramaggarwal/fashion-product-images-small"
)

print("✅ Fashion Product dataset downloaded.")
print("Path:", fashion_products_path)

Using Colab cache for faster access to the 'fashion-product-images-small' dataset.
✅ Fashion Product dataset downloaded.
Path: /kaggle/input/fashion-product-images-small


In [61]:
# =========================================================
# INSPECT FASHION PRODUCT DATASET
# =========================================================

for root, dirs, files in os.walk(fashion_products_path):

    print(f"\n📂 {root}")

    if len(files) > 0:
        print("Sample files:", files[:5])

    if len(dirs) > 0:
        print("Subfolders:", dirs[:5])

    if root.count(os.sep) > 3:
        break


📂 /kaggle/input/fashion-product-images-small
Sample files: ['styles.csv']
Subfolders: ['myntradataset', 'images']

📂 /kaggle/input/fashion-product-images-small/myntradataset
Sample files: ['styles.csv']
Subfolders: ['images']


In [62]:
# =========================================================
# INSPECT ECOMMERCE CATEGORY FOLDERS
# =========================================================

ecommerce_dataset_dir = os.path.join(
    ecommerce_path,
    "dataset_clean"
)

categories = sorted(os.listdir(ecommerce_dataset_dir))

print("✅ Ecommerce Categories:\n")

for category in categories:
    print(category)

✅ Ecommerce Categories:

casual_shirts
formal_pants
formal_shirts
jeans
men_cargos
printed_hoodies
printed_tshirts
solid_tshirts


In [63]:
# =========================================================
# CREATE UNIFIED LABEL MAP
# =========================================================

LABEL_MAP = {

    # Shirts
    "casual_shirts": "shirt",
    "formal_shirts": "shirt",

    # T-Shirts
    "printed_tshirts": "tshirt",
    "solid_tshirts": "tshirt",

    # Pants
    "formal_pants": "pants",
    "men_cargos": "pants",

    # Standalone
    "jeans": "jeans",
    "printed_hoodies": "hoodie"
}

print("✅ Unified Label Map Created:\n")

for k, v in LABEL_MAP.items():
    print(f"{k}  -->  {v}")

✅ Unified Label Map Created:

casual_shirts  -->  shirt
formal_shirts  -->  shirt
printed_tshirts  -->  tshirt
solid_tshirts  -->  tshirt
formal_pants  -->  pants
men_cargos  -->  pants
jeans  -->  jeans
printed_hoodies  -->  hoodie


In [64]:
# =========================================================
# COPY ECOMMERCE DATASET TO GOOGLE DRIVE
# =========================================================

import shutil

source_dir = os.path.join(
    ecommerce_path,
    "dataset_clean"
)

destination_dir = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "ecommerce",
    "dataset_clean"
)

if not os.path.exists(destination_dir):

    shutil.copytree(
        source_dir,
        destination_dir
    )

print("✅ Ecommerce dataset copied to Drive.")
print(destination_dir)

✅ Ecommerce dataset copied to Drive.
/content/drive/MyDrive/PersonalFashionStylistV2/datasets/ecommerce/dataset_clean


In [65]:
# =========================================================
# BUILD ECOMMERCE METADATA CSV
# =========================================================

metadata = []

supported_ext = CONFIG["image_extensions"]

drive_ecommerce_dir = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "ecommerce",
    "dataset_clean"
)

for original_category in categories:

    category_path = os.path.join(
        drive_ecommerce_dir,
        original_category
    )

    simplified_label = LABEL_MAP[original_category]

    for image_name in os.listdir(category_path):

        image_path = os.path.join(category_path, image_name)

        # Skip non-image files
        if not image_name.lower().endswith(tuple(supported_ext)):
            continue

        metadata.append({

            "image_path": image_path,
            "original_label": original_category,
            "label": simplified_label,
            "source": "ecommerce"
        })

metadata_df = pd.DataFrame(metadata)

print("✅ Metadata DataFrame Created.")
print("\nTotal Images:", len(metadata_df))

metadata_df.head()

✅ Metadata DataFrame Created.

Total Images: 8625


,image_path,original_label,label,source
0,/content/drive/MyDrive/PersonalFashionStylistV...,casual_shirts,shirt,ecommerce
1,/content/drive/MyDrive/PersonalFashionStylistV...,casual_shirts,shirt,ecommerce
2,/content/drive/MyDrive/PersonalFashionStylistV...,casual_shirts,shirt,ecommerce
3,/content/drive/MyDrive/PersonalFashionStylistV...,casual_shirts,shirt,ecommerce
4,/content/drive/MyDrive/PersonalFashionStylistV...,casual_shirts,shirt,ecommerce


In [66]:
# =========================================================
# CHECK CLASS DISTRIBUTION
# =========================================================

print("✅ Simplified Label Distribution:\n")

print(
    metadata_df["label"].value_counts()
)

✅ Simplified Label Distribution:

label
shirt     2174
tshirt    2137
pants     2123
jeans     1135
hoodie    1056
Name: count, dtype: int64


In [41]:
# =========================================================
# REMOVE CORRUPT IMAGES
# =========================================================

valid_rows = []

for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):

    image_path = row["image_path"]

    try:
        img = Image.open(image_path)
        img.verify()

        valid_rows.append(row)

    except Exception:
        pass

clean_metadata_df = pd.DataFrame(valid_rows)

print("\n✅ Corrupt Image Cleaning Complete.")

print(f"\nOriginal Images: {len(metadata_df)}")
print(f"Valid Images: {len(clean_metadata_df)}")

100%|██████████| 8625/8625 [00:50<00:00, 172.32it/s]



✅ Corrupt Image Cleaning Complete.

Original Images: 8625
Valid Images: 8625


In [42]:
# =========================================================
# TRAIN / VALIDATION / TEST SPLIT
# =========================================================

train_df, temp_df = train_test_split(
    clean_metadata_df,
    test_size=(1 - CONFIG["train_ratio"]),
    stratify=clean_metadata_df["label"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=CONFIG["seed"]
)

print("✅ Dataset Split Complete.\n")

print(f"Train Size: {len(train_df)}")
print(f"Validation Size: {len(val_df)}")
print(f"Test Size: {len(test_df)}")

✅ Dataset Split Complete.

Train Size: 6037
Validation Size: 1294
Test Size: 1294


In [43]:
# =========================================================
# CHECK SPLIT DISTRIBUTION
# =========================================================

print("📌 TRAIN DISTRIBUTION\n")
print(train_df["label"].value_counts())

print("\n📌 VALIDATION DISTRIBUTION\n")
print(val_df["label"].value_counts())

print("\n📌 TEST DISTRIBUTION\n")
print(test_df["label"].value_counts())

📌 TRAIN DISTRIBUTION

label
shirt     1522
tshirt    1496
pants     1486
jeans      794
hoodie     739
Name: count, dtype: int64

📌 VALIDATION DISTRIBUTION

label
shirt     326
tshirt    321
pants     318
jeans     171
hoodie    158
Name: count, dtype: int64

📌 TEST DISTRIBUTION

label
shirt     326
tshirt    320
pants     319
jeans     170
hoodie    159
Name: count, dtype: int64


In [44]:
# =========================================================
# SAVE METADATA CSV FILES
# =========================================================

metadata_dir = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "metadata"
)

# Save CSVs
train_df.to_csv(
    os.path.join(metadata_dir, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(metadata_dir, "val.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(metadata_dir, "test.csv"),
    index=False
)

print("✅ Train/Val/Test CSVs saved.")

✅ Train/Val/Test CSVs saved.


In [45]:
# =========================================================
# SAVE LABEL MAP
# =========================================================

label_map_path = os.path.join(
    PROJECT_ROOT,
    "models",
    "label_maps",
    "label_map.json"
)

with open(label_map_path, "w") as f:
    json.dump(LABEL_MAP, f, indent=4)

print("✅ Label map saved.")
print("Path:", label_map_path)

✅ Label map saved.
Path: /content/drive/MyDrive/PersonalFashionStylistV2/models/label_maps/label_map.json


In [46]:
# =========================================================
# CREATE CLASS INDEX MAPPING
# =========================================================

unique_labels = sorted(
    clean_metadata_df["label"].unique()
)

CLASS_TO_IDX = {
    label: idx
    for idx, label in enumerate(unique_labels)
}

IDX_TO_CLASS = {
    idx: label
    for label, idx in CLASS_TO_IDX.items()
}

print("✅ Class Index Mapping Created.\n")

print("CLASS_TO_IDX:")
print(CLASS_TO_IDX)

✅ Class Index Mapping Created.

CLASS_TO_IDX:
{'hoodie': 0, 'jeans': 1, 'pants': 2, 'shirt': 3, 'tshirt': 4}


In [47]:
# =========================================================
# SAVE CLASS INDEX MAPPINGS
# =========================================================

mapping_dir = os.path.join(
    PROJECT_ROOT,
    "models",
    "label_maps"
)

with open(os.path.join(mapping_dir, "class_to_idx.json"), "w") as f:
    json.dump(CLASS_TO_IDX, f, indent=4)

with open(os.path.join(mapping_dir, "idx_to_class.json"), "w") as f:
    json.dump(IDX_TO_CLASS, f, indent=4)

print("✅ Class mappings saved.")

✅ Class mappings saved.


In [48]:
# =========================================================
# LOAD FASHION PRODUCT METADATA
# =========================================================

fashion_csv_path = os.path.join(
    fashion_products_path,
    "styles.csv"
)

fashion_df = pd.read_csv(
    fashion_csv_path,
    on_bad_lines='skip'
)

print("✅ Fashion Product Metadata Loaded.")

print("\nShape:")
print(fashion_df.shape)

fashion_df.head()

✅ Fashion Product Metadata Loaded.

Shape:
(44424, 10)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt


In [49]:
# =========================================================
# CLEAN FASHION PRODUCT METADATA
# =========================================================

# Keep only important columns
important_cols = [

    "id",
    "gender",
    "masterCategory",
    "subCategory",
    "articleType",
    "baseColour",
    "season",
    "usage"
]

fashion_df = fashion_df[important_cols]

# Drop missing values
fashion_df = fashion_df.dropna()

print("✅ Cleaned Fashion Metadata.")

print("\nRemaining Rows:")
print(len(fashion_df))

✅ Cleaned Fashion Metadata.

Remaining Rows:
44079


In [50]:
# =========================================================
# ATTACH IMAGE PATHS
# =========================================================

fashion_images_dir = os.path.join(
    fashion_products_path,
    "images"
)

valid_rows = []

for idx, row in tqdm(fashion_df.iterrows(), total=len(fashion_df)):

    image_id = str(row["id"])

    image_path = os.path.join(
        fashion_images_dir,
        f"{image_id}.jpg"
    )

    if os.path.exists(image_path):

        row["image_path"] = image_path

        valid_rows.append(row)

fashion_inventory_df = pd.DataFrame(valid_rows)

print("\n✅ Fashion Inventory Ready.")

print(f"\nTotal Valid Fashion Images: {len(fashion_inventory_df)}")

100%|██████████| 44079/44079 [01:32<00:00, 475.71it/s]



✅ Fashion Inventory Ready.

Total Valid Fashion Images: 44074


In [51]:
# =========================================================
# SAVE FASHION INVENTORY METADATA
# =========================================================

inventory_metadata_path = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "metadata",
    "fashion_inventory.csv"
)

fashion_inventory_df.to_csv(
    inventory_metadata_path,
    index=False
)

print("✅ Fashion Inventory Metadata Saved.")
print("Path:", inventory_metadata_path)

✅ Fashion Inventory Metadata Saved.
Path: /content/drive/MyDrive/PersonalFashionStylistV2/datasets/metadata/fashion_inventory.csv


In [52]:
# =========================================================
# CREATE SMALL DEBUG SAMPLE
# =========================================================

sample_inventory_df = fashion_inventory_df.sample(
    n=1000,
    random_state=CONFIG["seed"]
)

sample_path = os.path.join(
    PROJECT_ROOT,
    "datasets",
    "metadata",
    "fashion_inventory_sample.csv"
)

sample_inventory_df.to_csv(
    sample_path,
    index=False
)

print("✅ Sample Inventory Saved.")
print("Rows:", len(sample_inventory_df))

✅ Sample Inventory Saved.
Rows: 1000
